In [1]:
# Standard includes
from pathlib import Path
import json
import re
from collections import defaultdict

# LLM / AI related libraries
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.language_models import ModelProfile
import torch
device = "mps" if torch.backends.mps.is_available() else "cpu"
from langchain_ollama import ChatOllama


# Local libraries
SOURCE_DIR = Path('cwd').parent.parent / "src" / "christmas_puzzle"
from christmas_puzzle.lyrics import raw_songs
TOP_K = 25

In [2]:
# Helpers

song_tags = {
    "carol_of_the_bells":["bells", "repetitive_pattern", "choral"],
    "little_drummer_boy": ["drums", "nonsense_syllables"],
    "deck_the_halls": ["fa_la_la", "choral", "cheerful"],
}

def normalize_sounds(line: str) -> str:
    line = re.sub(r"(ding[\s,-]*dong)+", "<BELL_SOUND>", line, flags=re.I)
    line = re.sub(r"(pa\s+rum\s+pum\s+pum\s+pum)+", "<DRUM_PATTERN>", line, flags=re.I)
    line = re.sub(r"(fa\s+la\s+la[^a-z]*)+", "<FA_LA_LA>", line, flags=re.I)
    return line


def slugify(title: str) -> str:
    s = title.lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def normalize_songs(raw_songs):
    cleaned = []
    for song in raw_songs:
        title = song["title"].strip()
        lyrics = song["lyrics"] or ""

        # Split on newlines, strip whitespace, drop empty lines
        lines = [
            line.strip()
            for line in lyrics.splitlines()
            if line.strip()
        ]

        cleaned.append(
            {
                "id": slugify(title),
                "title": title,
                "lines": lines,
            }
        )
    return cleaned

songs = normalize_songs(raw_songs)
for song in songs:
    song["lines"] = [normalize_sounds(l) for l in song["lines"]]



In [3]:
len(songs)

50

### Design translator from Elizabethan to modern English

In [10]:


translator_llm = ChatOllama(
    model="mistral:latest",
    temperature=0.1,
)

TRANSLATOR_SYSTEM = """
You rewrite faux-Elizabethan holiday song clues into simple modern English.

Goals:
- Preserve the original meaning and scene.
- Use concise, natural phrasing.
- Normalize repeated “nonsense” or sound effects into special tokens, but ONLY when they already appear in the original text.

Token rules:
Normalize repeated “nonsense” or sound effects into special tokens, but ONLY when they are clearly being used as sounds (not normal words).

Bell sounds → <BELL_SOUND>
- If the original text contains a string of repeated, non-word syllables that are clearly used
  as the sound of bells (for example, short syllables repeated around mentions of bells,
  ringing, chimes, or caroling), rewrite that sound string as <BELL_SOUND>.
- Examples of this kind of pattern include things like “ding dong”, “din don”, “ting ting”,
  “ching ching”, or other repetitive syllables that don’t form normal language.
- If you are not confident the syllables are being used as a bell sound, do NOT convert them.

Drum sounds → <DRUM_PATTERN>
- Same idea: repeated short syllables used as drum sounds (e.g. “pa rum pum pum”, “rum pum pum pum”)
  may be rewritten as <DRUM_PATTERN>.

Fa-la-la refrains → <FA_LA_LA>
- Repeated “fa / la” patterns that are obviously song filler rather than meaningful words
  may be rewritten as <FA_LA_LA>.

Hard constraints:
- Never invent <BELL_SOUND>, <DRUM_PATTERN>, or <FA_LA_LA> if the original clue does not
  contain any obvious sound-effect syllables.
- Do not add extra token-only lines; only insert tokens inline where they replace those sound segments.

Hard constraints:
- Never invent <DRUM_PATTERN>, <BELL_SOUND>, or <FA_LA_LA> if the original clue does not contain
  a corresponding sound-like phrase.
- Do not add extra token-only lines. Only include tokens inline where they directly replace
  those sound segments in your modern rewrite.

Hard constraints:
- NEVER invent <DRUM_PATTERN>, <BELL_SOUND>, or <FA_LA_LA> if the original clue
  does not contain a corresponding sound phrase.
- Do NOT add extra tokens or commentary after your translation.
- Output only 1–2 sentences of modern English, including tokens only where
  they correspond directly to sounds from the original text.
""".strip()

translator_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", TRANSLATOR_SYSTEM),
        ("user", "Rewrite this clue in modern English:\n\n{hint}"),
    ]
)

translator_llm = ChatOllama(
    model="mistral:latest",
    temperature=0.1,
)

translator_chain = translator_prompt | translator_llm

import re

TOKEN_PATTERN = re.compile(r"<[A-Z_]+>")

def clean_translation(hint: str, modern: str) -> str:
    """
    - Remove lines that are *only* tokens (e.g. "<DRUM_PATTERN> <BELL_SOUND>")
    - Optionally strip tokens that have no support in the original hint.
    """
    h = hint.lower()
    lines = modern.splitlines()
    cleaned_lines = []

    # map tokens to patterns that justify them
    token_rules = {
        "<DRUM_PATTERN>": ["pa rum", "rum pum", "drum"],
        "<BELL_SOUND>": ["ding dong", "tintinnabulum", "bell", "bells", "sleigh bells"],
        "<FA_LA_LA>": ["fa la la"],
    }

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue

        # if line has no letters and only tokens, drop it
        has_letters = re.search(r"[A-Za-z]", stripped) is not None
        tokens = TOKEN_PATTERN.findall(stripped)

        if tokens and not has_letters:
            # pure token spam -> skip
            continue

        # if tokens exist, drop the ones not grounded in original hint
        for tok, patterns in token_rules.items():
            if tok in stripped and not any(p in h for p in patterns):
                stripped = stripped.replace(tok, "")

        # collapse extra spaces
        stripped = re.sub(r"\s+", " ", stripped).strip()
        if stripped:
            cleaned_lines.append(stripped)

    return "\n".join(cleaned_lines).strip()

def to_modern(hint: str) -> str:
  raw = translator_chain.invoke({"hint": hint}).content.strip()
  return clean_translation(hint, raw)


### Build the song index w/ sliding lyric windows

In [11]:

WINDOW_SIZE = 2  # or 3 if your lines tend to be super short

def build_song_docs(songs, window_size: int = WINDOW_SIZE):
    docs = []
    for song in songs:
        lines = song["lines"]
        for i in range(len(lines) - window_size + 1):
            window = " / ".join(lines[i : i + window_size])
            docs.append(
                Document(
                    page_content=window,
                    metadata={
                        "song_id": song["id"],
                        "title": song["title"],
                        "start_line": i,
                        "end_line": i + window_size - 1,
                    },
                )
            )
    return docs

docs = build_song_docs(songs, window_size=WINDOW_SIZE)

embed = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    # or any embedding model you like
)

store = FAISS.from_documents(docs, embed)
retriever = store.as_retriever(search_kwargs={"k": TOP_K})


### Build the reranker - pick the best answers from a group of suggestions

In [12]:
ranker_llm = ChatOllama(
    model="mistral:latest",
    temperature=0.0,   # as deterministic as possible
)
SYSTEM_BATCH_RANKER = """
You are matching a holiday song riddle to multiple lyric excerpts.

You will receive:
- A HINT (faux-Elizabethan paraphrase).
- A numbered LIST of lyric EXCERPTS (1, 2, 3, ...).

Your job: For each excerpt, score how likely it is that it comes from the SAME specific lyric idea as the hint.

Scoring (0–10):
- 0–2: Unrelated. Different scene or no clear connection.
- 3–5: Vague holiday vibe only (Christmas/winter/joy) but NOT the same scene.
- 6–8: Clearly related scene or idea. Same type of situation, but missing some details.
- 9–10: Very strong paraphrase of the SAME lyric:
  - Same concrete situation,
  - Same key objects or actions,
  - Same relationship between them.

Important:
- Do NOT give scores above 6 just for generic holiday words like “Christmas”, “snow”, “night”, “bells”, “joy”, “spirit”.
- Treat tokens like <BELL_SOUND>, <DRUM_PATTERN>, <FA_LA_LA> as strong clues when they appear in BOTH hint and excerpt, but do not invent connections that aren’t there.

Examples:

HINT:
"The winds outside are dreadful, but the fire inside makes me happy."

GOOD (score ≈ 9–10):
"Oh, the weather outside is frightful / But the fire is so delightful."
- Same outside vs inside contrast, same weather + fire scene.

OK BUT NOT GREAT (score ≈ 4–5):
"Comfort and joy / O tidings of comfort and joy."
- Holiday mood and joy, but no weather, no fire, no inside/outside contrast.

UNRELATED (score ≈ 0–2):
"Joy to the world, the Lord is come / Let earth receive her King."
- Same holiday, but completely different scene and meaning.

Output:
Return ONLY valid JSON with a top-level key "scores".
"scores" maps string indices like "1", "2", "3", ... to numeric scores between 0 and 10.
""".strip()



batch_ranker_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_BATCH_RANKER),
        (
            "user",
            "HINT:\n{hint}\n\n"
            "EXCERPTS:\n{numbered_excerpts}\n\n"
            "JSON only:"
        ),
    ]
)

batch_reranker_llm = ChatOllama(
    model="mistral:latest",
    temperature=0.0,
)

batch_rerank_chain = batch_ranker_prompt | batch_reranker_llm


def select_windows_per_song(candidates, max_to_rerank=25, per_song_limit=3):
    """
    candidates: list of docs sorted by embedding similarity
    returns: list of docs, preserving order, but at most `per_song_limit` per song
    """
    per_song_count = defaultdict(int)
    selected = []

    for doc in candidates:
        sid = doc.metadata["song_id"]
        if per_song_count[sid] >= per_song_limit:
            continue
        selected.append(doc)
        per_song_count[sid] += 1
        if len(selected) >= max_to_rerank:
            break

    return selected


def rerank_candidates_batch(hint: str, candidates, max_to_rerank: int = 25):
    docs = select_windows_per_song(
        candidates,
        max_to_rerank=max_to_rerank,
        per_song_limit=3,
    )

    lines = []
    for i, doc in enumerate(docs, start=1):
        snippet = doc.page_content.replace("\n", " / ")
        lines.append(f"{i}. {snippet}")
    numbered_excerpts = "\n".join(lines)

    raw = batch_rerank_chain.invoke(
        {"hint": hint, "numbered_excerpts": numbered_excerpts}
    ).content.strip()

    # Debug once so you can see the raw model output
    # print("RAW RERANK JSON:", raw)

    try:
        data = json.loads(raw)
    except Exception as e:
        print("JSON parse error:", e)
        data = {}

    # Handle either:
    #   {"scores": {...}}  OR  {"1": 9, "2": 4, ...}
    if isinstance(data, dict) and "scores" in data and isinstance(data["scores"], dict):
        scores_dict = data["scores"]
    elif isinstance(data, dict):
        scores_dict = data
    else:
        scores_dict = {}

    scored_docs = []
    for i, doc in enumerate(docs, start=1):
        score = float(scores_dict.get(str(i), 0.0))
        scored_docs.append((doc, score))

    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return scored_docs



In [13]:
def aggregate_by_song(scored_docs, top_songs: int = 3):
    best_score = {}
    best_doc = {}

    for doc, score in scored_docs:
        sid = doc.metadata["song_id"]
        if sid not in best_score or score > best_score[sid]:
            best_score[sid] = score
            best_doc[sid] = doc

    ranked = sorted(best_score.items(), key=lambda x: x[1], reverse=True)

    results = []
    for sid, s in ranked[:top_songs]:
        doc = best_doc[sid]
        results.append(
            {
                "song_id": sid,
                "title": doc.metadata["title"],
                "score": s,
                "sample_window": doc.page_content,
            }
        )
    return results



### Create the song guesser

In [14]:

def guess_song(elizabethan_hint: str, k_docs: int = 10, k_songs: int = 3):
    # 1) Normalize language
    modern = to_modern(elizabethan_hint)

    # 2) Retrieve lyric windows
    candidates = retriever.invoke(modern)  # already sorted by similarity

    # 3) Rerank candidates
    scored_docs = rerank_candidates_batch(modern, candidates, TOP_K)

    # 4) Aggregate score by song -- return top_songs 
    ranked_songs = aggregate_by_song(scored_docs, top_songs = 3)

    return {
        "modern_query": modern,
        "results": ranked_songs,
    }


### Run the puzzle

In [17]:
elizabethan_hints = ["O, the winds without are dreadful, Yet the fire within maketh me joyous.",
                   "Tolling tintinnabulum, tolling tintinnabulum—Ah! The rhythm of tolling tintinnabulum! The tintinnabulum doth swing and—Yea!—also dost ring",
                   """"Come!", they didst proclaim, pa rum pum pum pum
A newly wrought monarch to behold, pa rum pum pum pum""",
"""Santa, thou sweet babe, prithee slip a furred mantle 'neath the tree for me;
I have been a most virtuous maid.""",
"""Still night, hallow'd night,
All is hush'd, all is aglow.""",
"""Cover thy pate—Lo! Chanukah approacheth,
Such mirth and excit'ment-ukah, to make merry for Chanukah.""",
"""Thou art a churlish wight—verily, a knave art thou
Thou art as huggable as a prickly plant, as winsome as a serpent of the watery deep""",
"""So this be Yuletide, and what hast thou wrought?
Another year hath ended, and a new one hath but begun.""",
"""Old Hiems, with his black and icy crown, was a blithe and merry spirit,
With a pipe of maize, and a proboscis of button wrought, and twain eyes of ember.""",
"""The snow doth gleam as white upon the mountain's height,
As all who pass on foot go untrack'd""",
"""My heart longeth for a Yuletide clad in white,
E'en as those I knew in days of yore.""",
"""'Twas the four and twentieth of December upon Hollis Avenue after night's fall,
When I espied a gentleman reclining with his hound in the village green."""]

for hint in elizabethan_hints:
    out = guess_song(hint)
    print("Hint:",hint)
    print("Modern query:", out["modern_query"])
    for r in out["results"][:1]:
        print(f"- {r['title']} (score={r['score']})")
        print(f"  Match: {r['sample_window']}")
    print("\n")

Hint: O, the winds without are dreadful, Yet the fire within maketh me joyous.
Modern query: The cold wind outside is frightening, but I'm warm and happy inside.
Good Christian men, rejoice with hearty goodwill,
Let earth resound, yield heavenly peace!
Let every heart prepare him room,
And heav'n and nature sing, "Gloria!"
Gloria, in excelsis Deo!
- Joy to the World (score=10.0)
  Match: Let every heart prepare Him room, / And heaven and nature sing.


Hint: Tolling tintinnabulum, tolling tintinnabulum—Ah! The rhythm of tolling tintinnabulum! The tintinnabulum doth swing and—Yea!—also dost ring
Modern query: Bells are continuously ringing, creating a rhythmic sound. They sway back and forth while also producing a resonating tone.
- Carol of the bells (score=10.0)
  Match: <BELL_SOUND>, <BELL_SOUND> / Hark how the bells, sweet silver bells


Hint: "Come!", they didst proclaim, pa rum pum pum pum
A newly wrought monarch to behold, pa rum pum pum pum
Modern query: "Come here!" they announ